<a href="https://colab.research.google.com/github/tinawanggg/Wang_DSPN_S26/blob/master/book/exercises/12_cross-validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 12: Cross validation

In this exercise, we'll practice implementing cross validation techniques, including leave-one-out and k-fold cross validation. We'll use the `PimaIndiansDiabetes2` practice dataset, which has medical data on a group of Pima Native American women, including whether or not they have diabetes. This dataset is part of the `mlbench` package. We'll be using each person's medical history to predict whether or not they have been diagnosed with diabetes.

---
## 1. Load Data (1 point)

Load the `tidyverse`, `boot`, and `mlbench` packages (you may need to install `boot` and `mlbench`).

Load the `PimaIndiansDiabetes2` dataset using the `data()` function. Drop the `insulin` column (it just has a lot of missing data) and then drop `NA`s from the rest of the dataset. Save your updated dataset to a new variable name. Finally, print the dimensions of your new dataset, and look at the first few lines of data.

In [1]:
# INSERT CODE HERE
install.packages("boot")
install.packages("mlbench")

library(tidyverse)
library(boot)
library(mlbench)

data("PimaIndiansDiabetes2")

pima <- PimaIndiansDiabetes2 %>%
  select(-insulin) %>%
  drop_na()

dim(pima)
head(pima)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] 532   8

,pregnant,glucose,pressure,triceps,mass,pedigree,age,diabetes
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,6,148,72,35,33.6,0.627,50,pos
2,1,85,66,29,26.6,0.351,31,neg
4,1,89,66,23,28.1,0.167,21,neg
5,0,137,40,35,43.1,2.288,33,pos
7,3,78,50,32,31.0,0.248,26,pos
9,2,197,70,45,30.5,0.158,53,pos


(Note that in medical contexts, `pedigree` refers to a system of measuring family history of a condition. So here, higher numbers mean greater family history of diabetes. You can read more about this dataset [here](https://rdrr.io/cran/mlbench/man/PimaIndiansDiabetes.html).)

---
## 2. Leave-one-out Cross Validation (4 points)

In the tutorial, we learned how to fit leave-one-out cross validation using the `cv.glm` function from the `boot` package. But we can also do this manually using `predict()` like we have in the past.

Let's predict `diabetes`, a dichotomous outcome, using all the other variables in our modified dataset.

First, fit a logistic regression model using all of the observations except the very first one. Then use your fitted model to predict whether your holdout case is positive or negative for diabetes. Remember that in logistic regression, the model output (before applying the sigmoid function) is in **log-odds**. If this output is positive, the predicted probability is greater than 50%; if it is negative, the predicted probability is less than 50%.

Compare your prediction to the actual response for the first observation that you initially excluded. Did your model correctly classify this observation?

In [8]:
# INSERT CODE HERE
train <- pima[-1, ]
model_loo <- glm(diabetes ~ ., data = train, family = binomial)

holdout <- pima[1, ]
log_odds <- predict(model_loo, newdata = holdout)
log_odds
predicted_class <- ifelse(log_odds > 0, "pos", "neg")
predicted_class

actual_class <- as.character(holdout$diabetes)
actual_class

1 
0.9920213

1 
"pos"

[1] "pos"

*The model did correctly classify this observation. The actual response was 'pos,' and my model's classification was also positive.*

So we just calculated a single iteration of leave-one-out cross validation. We used 531 rows of our data to fit a model to predict the outcome of the last row.

Below, use a `for` loop to iterate through the rest of your dataset doing the same thing. You will need to:
* Create a data frame `results` with two columns: one named `actual` which holds the true classification for each observation, and one named `predicted`, which should be filled with `NA`s. This is where you'll store the output of your loop.
* Create a loop that runs through each row of your data, pulls that observation out, trains your model on the remaining data, and then tests the fitted model on your test observation.
* Store your model *predictions* ("pos" or "neg" -- not the log-odds) in the `predicted` column of your `results` dataframe

After you run your loop, print the first few lines of `results`.

In [16]:
# Initialize `results` data frame
# INSERT CODE HERE
results <- data.frame(
  actual    = pima$diabetes,
  predicted = NA
)

#for loop
for (i in 1:nrow(pima)){
    # separate individual observation `i` from the rest of your data
    # INSERT CODE HERE
    train   <- pima[-i, ]
    holdout <- pima[i, ]

    # train your model
    # INSERT CODE HERE
    model_loo <- glm(diabetes ~ ., data = train, family = binomial)

    # test model on hold out observation
    # INSERT CODE HERE
    log_odds <- predict(model_loo, newdata = holdout)

    # classify model prediction as "pos" or "neg" and add to `results`
    # INSERT CODE HERE
    results$predicted[i] <- ifelse(log_odds > 0, "pos", "neg")
}


head(results)

,actual,predicted
,<fct>,<chr>
1,pos,pos
2,neg,neg
3,neg,neg
4,pos,pos
5,pos,neg
6,pos,pos


Now, calculate the overall error of your model. What proportion of cases were incorrectly classified?

In [13]:
# INSERT CODE HERE
error_rate <- mean(results$predicted != results$actual)
error_rate

[1] 0.2218045

The error rate is approximately 0.2218, which suggests that 22.2% of cases were incorrectly classified. If there were 532 observations/cases, that means that about 118 of them were incorrectly classified.

----
## 3. Compare to `cv.glm` (3 points)

Now, let's compare this result to the `cv.glm` function. Using the tutorial as a guide, use `cv.glm` to run LOOCV on the data, using the same model (i.e. still using all of the variables to predict diabetes diagnosis).

Note that, because this is a `classification` problem and not a regression problem like in the tutorial, we need to adjust the `cost` argument of `cv.glm`. For more details, see the function documentation by running `?cv.glm`:

In [17]:
?cv.glm

Here, we see `cost` is defined as:
> "A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the **observed responses** and the second argument should correspond to the **predicted or fitted responses** from the generalized linear model."

In the example code (scroll to bottom of the docs), we see that the appropriate cost function for a binary classification is

```
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}
```

Where `r` is the vector of observed responses (technically "pos" and "neg", but R treats these as 1 and 0 under the hood), and `pi` is the vector of *probabilities* (not log-odds) fit by the model. Thus, this boils down to our error: what proportion of observations were incorrectly classified. You will need to include this code below.

In [18]:
# INSERT CODE HERE
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}

model_full <- glm(diabetes ~ ., data = pima, family = binomial)
loo_cv <- cv.glm(data = pima, glmfit = model_full, cost = cost, K = nrow(pima))
loo_cv$delta

[1] 0.2218045 0.2221154

How do your results compare to your manual LOOCV above?

> * *The error from this is also approximately 0.2218, or 22.2%. This is almost an exact match to the earlier manual LOOCV error rate of 22.2%.*


---
## 4. Adjusting K and Reflection (2 points)

Recall that LOOCV has some drawbacks. In particular, it has quite high *variance* which can lead to poor performance on new test data. We can reduce this variance by increasing K.

Below, re-run your cross validation using `cv.glm` with `k` set to 3, 5, 10, and 15.

In [23]:
set.seed(1)
#INSERT CODE BELOW

# K = 3
cv_3 <- cv.glm(data = pima, glmfit = model_full, cost = cost, K = 3)
cv_3$delta[1]

# K = 5
cv_5 <- cv.glm(data = pima, glmfit = model_full, cost = cost, K = 5)
cv_5$delta[1]

# K = 10
cv_10 <- cv.glm(data = pima, glmfit = model_full, cost = cost, K = 10)
cv_10$delta[1]

# K = 15
cv_15 <- cv.glm(data = pima, glmfit = model_full, cost = cost, K = 15)
cv_15$delta[1]

[1] 0.2105263

[1] 0.2161654

[1] 0.2236842

[1] 0.2274436

#### Reflection

How do your errors compare to your LOOCV error above? How do they change as k increases?
> * *It seems like the generated errors are still very close to the LOOCV error of 22.2%, with small variations as k increases. It appears that the error rates are increasing as k increases.*


If you change the random seed above, you'll get slightly different errors. If you were to do the same with your LOOCV above , would you expect to get different results each time? Why or why not?
> * *No, changing random seeds for the LOOCV should lead to the same result every time. This is because setting a random seed while increasing k introduces randomness in different groupings of data, but there are no different groupings in LOOCV. LOOCV involves going in order of holding out the first observation, then training on the rest, then holding out on the second observation, and so on. So in other words, there is only one order followed by LOOCV depending on the given data.*

**DUE:** 11:59pm March 19, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *N/A*
>
>

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

>
>